In [1]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Grounding testing

### Installation

Install the following packages required to execute this notebook.

In [2]:
%pip install --upgrade google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.


Restart the kernel after installing packages:

In [3]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

### Configure your project ID

**If you don't know your project ID**, try the following:
* Run `gcloud config list`.
* Run `gcloud projects list`.
* See the support page: [Locate the project ID](https://support.google.com/googleapi/answer/7014113)

In [2]:
PROJECT_ID = "mandieq-demo"  # @param {type:"string"}

# Set the project ID
!gcloud config set project {PROJECT_ID}

Updated property [core/project].


### Configure your region

You can also change the `REGION` variable used by Vertex AI. Learn more about [Vertex AI regions](https://cloud.google.com/vertex-ai/docs/general/locations).

In [3]:
REGION = "us-central1"  # @param {type: "string"}

### Import libraries

In [58]:
import vertexai
from vertexai.language_models import TextGenerationModel, ChatModel, GroundingSource

from vertexai.preview.generative_models import (
    GenerationResponse,
    GenerativeModel,
    GenerationConfig,
    grounding,
    Tool,
)

from IPython.display import display, Markdown

### Initialize Vertex AI SDK for Python

Initialize the Vertex AI SDK for Python for your project:

In [6]:
vertexai.init(project=PROJECT_ID, location=REGION)

Initialize the generative text and chat models from Vertex AI:

In [31]:
text_model = TextGenerationModel.from_pretrained("text-bison")
chat_model = ChatModel.from_pretrained("chat-bison")

text_model_gemini = GenerativeModel(model_name="gemini-1.0-pro")

## Using Vertex AI Search datastore

### Creating a data store in Vertex AI Search

Follow the steps in the [Vertex AI Search getting started documentation](https://cloud.google.com/generative-ai-app-builder/docs/try-enterprise-search#create_a_search_app_for_website_data) to create a data store in Vertex AI Search with sample data. In this example, you'll use a website-based data store that contains content from the Google Cloud website, including documentation.

Once you've created a data store, obtain the Data Store ID and input it below.

<div class="alert alert-block alert-warning">
Important - ensure that chunking isn't turned on for your Data Store.
</div>

MQ note - using a datastore consisting of one document only for testing:

https://www.dunnhumby.com/wp-content/uploads/reports/The_7_drivers_of_Price_Perception___how_you_can_influence_them_compressed.pdf 

In [25]:
DATA_STORE_ID = "project-dh-grounding2_1710852066159" # Replace this with your data store ID from Vertex AI Search
DATA_STORE_REGION = "global"

Now you can ask a question about object tables in BigQuery and when to use them:

In [9]:
PROMPT = "What are the drivers of price perception?"

## Text-bison

### Text generation without grounding

Make a prediction request to the LLM with no grounding:

In [21]:
response = text_model.predict(PROMPT,
                              max_output_tokens=1024)

# response, response.grounding_metadata

display(Markdown(response.text))

print(response.grounding_metadata)

 Price perception is influenced by various factors that shape consumers' understanding and evaluation of prices. Here are some key drivers of price perception:

1. **Reference Prices:** Consumers often compare prices to a reference point, which could be a previous price, a competitor's price, or a perceived fair price. If the current price is lower than the reference price, it may be perceived as a good deal.

2. **Product Quality:** Consumers associate higher prices with higher quality. If a product is perceived to be of high quality, consumers may be willing to pay a premium price for it.

3. **Brand Image:** Well-established brands often command higher prices due to their reputation and perceived value. Consumers may be willing to pay more for a branded product because they trust the brand's quality and reliability.

4. **Exclusivity and Scarcity:** Limited edition or exclusive products may be perceived as more valuable and desirable, leading to higher price tags. Scarcity can create a sense of urgency and increase willingness to pay.

5. **Discounts and Promotions:** Discounts, sales, and special offers can significantly impact price perception. Consumers may perceive a product as being more affordable or a better value when it is on sale.

6. **Context and Environment:** The context in which a product is presented can influence price perception. For example, a product displayed in a luxurious setting may be perceived as more expensive than the same product in a budget-friendly environment.

7. **Consumer Expectations:** Consumers' expectations about the price of a product can shape their perception. If they expect a product to be expensive, they may be less sensitive to its actual price.

8. **Social Norms and Cultural Factors:** Cultural norms and social influences can impact price perception. In some cultures, paying a higher price may be seen as a sign of status or prestige.

9. **Personal Factors:** Individual factors such as income, financial situation, and personal values can influence price perception. Consumers with higher disposable incomes may be less sensitive to prices compared to those with limited financial resources.

10. **Marketing and Advertising:** Marketing strategies, advertising, and product packaging can influence price perception. Effective marketing can create a positive image of a product and make consumers more receptive to higher prices.

Understanding these drivers of price perception is crucial for businesses to effectively set prices, communicate value, and influence consumer behavior. By considering these factors, companies can optimize their pricing strategies and create a positive perception of their products or services in the market.

GroundingMetadata(citations=[], search_queries=[])


### Text generation grounded in Vertex AI Search results

Now we can add the `grounding_source` keyword arg with a grounding source of `GroundingSource.VertexAISearch()` to instruct the LLM to first perform a search within your custom data store, then construct an answer based on the relevant documents:

In [27]:
grounding_source = GroundingSource.VertexAISearch(
    data_store_id=DATA_STORE_ID, location=DATA_STORE_REGION
)

response = text_model.predict(
    PROMPT,
    max_output_tokens=1024,
    temperature=0.1,
    grounding_source=grounding_source,
)

# response, response.grounding_metadata

display(Markdown(response.text))

print(response.grounding_metadata)

 The seven drivers of price perception are:

1. **Reference price**: This is the price that the customer expects to pay for a product or service. It can be based on past purchases, prices seen in advertising, or prices charged by competitors.
2. **Perceived quality**: This is the customer's perception of the quality of a product or service. It can be based on factors such as brand reputation, product features, and customer reviews.
3. **Perceived value**: This is the customer's perception of the benefits of a product or service relative to its price. It can be based on factors such as the product's features, quality, and price.
4. **Purchase situation**: This refers to the circumstances in which the customer is making a purchase. It can be based on factors such as the time of day, the location of the purchase, and the customer's mood.
5. **Social norms**: This refers to the prices that other people are paying for a product or service. It can be based on factors such as the prices charged by competitors, the prices paid by friends and family, and the prices advertised in the media.
6. **Marketing communications**: This refers to the messages that the customer receives about a product or service from the company. It can be based on factors such as advertising, sales promotions, and public relations.
7. **Personal factors**: This refers to the individual characteristics of the customer that influence their price perception. It can be based on factors such as the customer's age, gender, income, and personality.

GroundingMetadata(citations=[GroundingCitation(start_index=1187, end_index=1255, url='gs://mq-project-dh/The_7_drivers_of_Price_Perception___how_you_can_influence_them_compressed.pdf', title='The_7_drivers_of_Price_Perception___how_you_can_influence_them_compressed', license=None, publication_date=None)], search_queries=['What are the drivers of price perception?'])


## Gemini

### Text generation without grounding

Make a prediction request to the LLM with no grounding:

In [32]:
response = text_model_gemini.generate_content(PROMPT)

# response, response.grounding_metadata

display(Markdown(response.text))

# print(response.grounding_metadata)

**Internal Factors:**

* **Product knowledge and experience:** Familiarity with the product or similar products influences price expectations.
* **Personal values and beliefs:** Consumers' beliefs about the worth and importance of the product shape their price perceptions.
* **Financial resources:** Income, wealth, and spending habits influence consumers' willingness to pay.
* **Psychological factors:** Emotions, motivations, and expectations can affect price perception. For example, consumers may be willing to pay more for products that evoke positive emotions.
* **Cognitive biases:** Mental shortcuts and heuristics can influence how consumers perceive prices. For example, the "anchoring effect" refers to the tendency to use the first price encountered as a reference point for subsequent prices.

**External Factors:**

* **Market characteristics:** Competition, supply and demand, and industry norms influence price expectations.
* **Marketing communications:** Advertising, promotions, and public relations efforts can shape consumers' perceptions of product value and price.
* **Social influences:** Friends, family, and peers can communicate their price expectations and influence consumers' own perceptions.
* **Cultural and socioeconomic factors:** Cultural norms, social class, and economic conditions can influence price sensitivity.
* **Environmental cues:** The physical environment, such as store decor and product packaging, can provide signals about price.
* **Product cues:** Physical attributes of the product, such as size, shape, and materials, can influence perceived value and price.
* **Price framing:** The way prices are presented, such as using reference prices or discounts, can affect consumers' perceptions.
* **Availability and scarcity:** The perceived availability or scarcity of a product can influence its perceived value and price.

In [37]:
response.to_dict()

{'candidates': [{'content': {'role': 'model',
    'parts': [{'text': '**Internal Factors:**\n\n* **Product knowledge and experience:** Familiarity with the product or similar products influences price expectations.\n* **Personal values and beliefs:** Consumers\' beliefs about the worth and importance of the product shape their price perceptions.\n* **Financial resources:** Income, wealth, and spending habits influence consumers\' willingness to pay.\n* **Psychological factors:** Emotions, motivations, and expectations can affect price perception. For example, consumers may be willing to pay more for products that evoke positive emotions.\n* **Cognitive biases:** Mental shortcuts and heuristics can influence how consumers perceive prices. For example, the "anchoring effect" refers to the tendency to use the first price encountered as a reference point for subsequent prices.\n\n**External Factors:**\n\n* **Market characteristics:** Competition, supply and demand, and industry norms influ

### Text generation grounded in Vertex AI Search results

Now we can add the `grounding_source` keyword arg with a grounding source of `GroundingSource.VertexAISearch()` to instruct the LLM to first perform a search within your custom data store, then construct an answer based on the relevant documents:

In [71]:
data_store_path = f"projects/{PROJECT_ID}/locations/global/collections/default_collection/dataStores/{DATA_STORE_ID}"

tool = Tool.from_retrieval(
    grounding.Retrieval(grounding.VertexAISearch(datastore=data_store_path))
    )

response = text_model_gemini.generate_content(PROMPT,
                                              tools=[tool])

display(Markdown(response.text))

# print(response.to_dict()['candidates'][0]['grounding_metadata'])

for cand in response.to_dict()['candidates']:
    print(cand['grounding_metadata'])

The seven drivers of price perception are:

* **Price:** The actual price of the product.
* **Reference prices:** The prices of similar products that the customer is aware of.
* **Promotions:** Sales, discounts, and other promotions that can affect the customer's perception of the price.
* **Brand:** The reputation and image of the brand can influence the customer's perception of the price.
* **Store:** The type of store where the product is purchased can affect the customer's perception of the price.
* **Context:** The situation in which the purchase is made can affect the customer's perception of the price.
* **Personal factors:** The customer's individual beliefs, attitudes, and experiences can affect their perception of the price.

{'web_search_queries': ['What are the drivers of price perception?'], 'grounding_attributions': [{'segment': {'start_index': 394, 'end_index': 506, 'part_index': 0}, 'confidence_score': 0.7781065, 'web': {'uri': 'gs://mq-project-dh/The_7_drivers_of_Price_Perception___how_you_can_influence_them_compressed.pdf', 'title': 'The_7_drivers_of_Price_Perception___how_you_can_influence_them_compressed'}}]}
